# Meqpy Tutorial — 1. Rate Equation: Basics

[🏠 Index](00_Overview.ipynb) | [Next: 2. System and States →](02_System_and_States.ipynb)

- [1. Rate Equation: Basics](#basics)
    - [1.1 Solve for Equilibrium State](#basics_solve)
    - [1.2 Examples](#basics_example)
        - [1.2.1 Simple](#basics_example_simple)
        - [1.2.2 Choosing Anchor](#basics_example_anchor)
        - [1.2.3 Adding Fuzz](#basics_example_fuzz)


In [ ]:
import meqpy
import numpy as np

<a id='basics'></a>
## 1. Rate Equation: Basics

The goal of the ``meqpy`` package is to provide an easy and fast way to perform rate equation simulations, with a special focus on transport experiments such as scanning tunneling microscopy (STM) and spectroscopy (STS).

The investigated system is constructed in the form of a Markov chain with a number of distinct states and certain transition rates between those states. Assuming a system with $n$ states, the probability of finding the system in state $i$ is defined as $P_i$ in the occupation probability vector $\mathbf{P}$. For consistency, the sum of all probabilities must satisfy $\sum_i P_i = 1$.

The transition rates $\Gamma$ between the individual states are written in the rate matrix $\mathbf{M}$, with $M_{fi} = \Gamma_{i \rightarrow f}$. The diagonal elements of the rate matrix are defined as $M_{ii} = - \sum_{i \neq f} \Gamma_{i \rightarrow f}$.

The rate equation is a first-order differential equation and can be written as a master equation:
```math
\frac{d}{dt}\mathbf{P} = \mathbf{M} \cdot \mathbf{P}
```

STM/STS experiments are usually performed in the equilibrium regime, where changes in the parameters are assumed to occur on a much slower timescale than the transition rates. Thus, the system will be in a steady state $P^\mathrm{eq}$ with:
```math
\mathbf{0} = \mathbf{M} \cdot \mathbf{P^{eq}}
```
The ``meqpy`` package provides all the necessary tools to construct the rate matrix $\mathbf{M}$ for STM/STS experiments and to solve the master equation in the equilibrium regime.


<a id='basics_solve'></a>
### 1.1 Solve for Equilibrium State

For fast batch calculation of matrix arrays with shape ``(..., N, N)`` the ``scipy.linalg.solve`` method is used, which can solve ``a @ x = b`` for ``x``. However, in this case ``x`` is not well defined since ``b`` is the null vector and ``x`` can accordingly be rescaled by any real number to satisfy the equilibrium equation.

To circumvent this problem, while maintaining fast batch calculation, ``meqpy`` uses one state $a$ in the rate matrix $\mathbf{M}$ as anchor, defines $P^{eq}_a = 1$ and solves the remaining of the master equation accordingly.

As an example, let us assume a $3 \times 3$ rate matrix:
```math
   \begin{pmatrix}0 \\\ 0 \\\ 0 \end{pmatrix}
    =  \begin{pmatrix} 
    a & b & c \\\ 
    d & e & f \\\ 
    g & h & j
   \end{pmatrix}
   \cdot
   \mathbf{x}
```

We use the first state as anchor and define $x_0 = 1$. The new reduced master equation is then:
```math
   \begin{pmatrix} -d \\\ -g \end{pmatrix}
    =  \begin{pmatrix} 
    e & f \\\ 
    h & j
   \end{pmatrix}
   \cdot
   \mathbf{x'},
```
which (should) have a well defined solution. This method is much faster than SVD based methods like ``scipy.linalg.null_space`` but it comes with some caveats which need to be considered, as we will discuss later.

As a last step, the anchor state is inserted back into $x'$ and $x$ is being renormed to satisfy $\sum_i P_i = 1$.

<a id='basics_example'></a>
### 1.2 Examples

The steps described above are all handled by the ``meqpy.solve_equilibrium`` method. It checks that the input matrix is of shape ``(..., N, N)`` with non-negative values in the off-diagonal elements $M_{i \neq f}$. The diagonal is then filled automatically with  $M_{ii} = - \sum_{i \neq f} \Gamma_{i \rightarrow f}$.

``meqpy.solve_equilibrium`` takes four inputs:
- ``W``: array of shape ``(..., N, N)`` to be solved
- ``tol``: tolerance for negative values in $P^{eq}$, default is ``1e-12``
- ``anchor``: which state to use as the anchor, default is ``0`` (first state)
- ``fuzz``: constant rate added to all transitions, relative to the maximum of each ``(N, N)`` submatrix, default is ``0``

The method returns an array of shape ``(..., N)``, containing the occupation probability vector $P^\mathrm{eq}$ for each ``(N, N)`` submatrix.

The following provides some simple examples of how to use the ``meqpy.solve_equilibrium`` method and what to look out for.

<a id='basics_example_simple'></a>
#### 1.2.1 Simple Case
Let's assume a simple three state system, where the first state is the ground state and the two other states have a transition rate of 1 to the ground state. This system is described in a $3\times 3$ matrix $W$ with all entries being zero, except for ``W[0,1] = W[0,2] = 1``, with ``W[f,i]`` corresponding to the rate from initial state ``i`` to final state ``f``. We will not assume any transition rate out of the ground state (``W[1:,0] = 0``), hence we expect a solution where the occupation probability of the first state is 1 and all others are 0.

Since ``meqpy.solve_equilibrium`` will fill the diagonal entries automatically, they can be left as is.

In [ ]:
# create rate matrix
W = np.zeros((3, 3))
W[0, 1:] = 1.0  # add transitions
W

In [ ]:
# solve matrix for equilibrium condition
meqpy.solve_equilibrium(W)

As described above, ``meqpy.solve_equilibrium`` does also work with batches of square matrices

In [ ]:
# create four 3x3 matrices
W_batch = np.arange(36).reshape(4, 3, 3)
W_batch.shape

In [ ]:
# solve to obtain four vectors of length 3
meqpy.solve_equilibrium(W_batch).shape

<a id='basics_example_anchor'></a>
#### 1.2.2 Choosing Anchor
As described above, the default anchor state for solving the rate matrix is the first state by index. However, if the anchor state has an occupation probability of 0, this method will fail.
For example, we can use the same system as described above but we switch the first and second state. Now the first and last state will transition into the second state, which is the ground state:

In [ ]:
W_anchor = np.zeros((3, 3))
W_anchor[1, 0::2] = 1.0
W_anchor

Trying to solve this matrix while using the first state as an anchor will fail and raise a ``ValueError``:

In [ ]:
try:
    meqpy.solve_equilibrium(W_anchor)
except ValueError as e_info:
    print(str(e_info))

The simple solution is to choose the second state as an anchor.

In [ ]:
meqpy.solve_equilibrium(W_anchor, anchor=1)

It is best to always use a state with an expected non-zero occupation probability as the anchor state. Since ``anchor = 0`` is the default setting, it is recommended to define your system such that the first state corresponds to the ground state.

<a id='basics_example_fuzz'></a>
#### 1.2.3 Adding Fuzz
In some cases, there may be no suitable anchor state — either because the system is in a bistable regime, or because different matrices within a batch calculation may require different anchors. In these cases, a possible workaround is to add a very small background noise to all transitions, ensuring that all states have a non-zero occupation probability. This can be done via the ``fuzz`` parameter, which adds a constant value to all transitions in each submatrix. The added value is ``fuzz`` times the maximum value of the submatrix.

As an example, we again consider a system with three states, but this time the third state transitions with equal probability into either the first or the second state. Since there is no transition out of those two states, any vector ($P_0$, $P_1$, 0) with $P_0 + P_1 = 1$ is a valid solution, and a single anchor state is not sufficient to solve the problem.

In [ ]:
W_undefined = np.zeros((3, 3))
W_undefined[:2, 2] = 1.0
W_undefined

In [ ]:
# trying all possible anchor states
for i in range(3):
    try:
        meqpy.solve_equilibrium(W_undefined, anchor=i)
    except ValueError:
        print(f"anchor = {i} does not work.")

Using the ``fuzz`` parameter, a valid solution can be obtained for the system, even though it may not be the only solution.

In [ ]:
meqpy.solve_equilibrium(W_undefined, fuzz=1e-20)

---

[🏠 Index](00_Overview.ipynb) | [Next: 2. System and States →](02_System_and_States.ipynb)